# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane: Lane 4 — CTR / Engagement Opportunity Scoring** (locked at the end of Week 4).

**The question, in plain words:** looking only at what I could know at the end of February,
which pages should an editor review first, because they are about to under-capture clicks in March?

**What this notebook has to prove:** that a learned model beats the transparent rule I froze in
Week 4 — on the same rows, the same folds, the same metric, and the same tie policy. If it does
not beat the rule, that is the finding and I report it.


## Setup — connect and pull the two windows

*Features from January + February. Label from March. June 2026 is the sealed final month and is never queried.*


In [16]:
# ---- Setup: same warehouse connection as ML-04 / ML-07 ----
import duckdb, os, json, numpy as np, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # Colab Secret. Never pasted in a cell.
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")
con.sql("SET preserve_insertion_order=false;")

SEED = 42
rng  = np.random.default_rng(SEED)

BASE        = "hf://datasets/FlyRank/internship-warehouse"
FEAT_MONTHS = ["2026-01", "2026-02"]   # everything knowable BEFORE the decision
LABEL_MONTH = "2026-03"                # the outcome window
SEALED      = "2026-06"                # final panel month - deliberately never touched

FLOOR_FEB   = 500    # a page must be genuinely visible in Feb to be worth an editor's time
FLOOR_MAR   = 100    # and must have enough March traffic for its March CTR to mean anything

os.makedirs("work/outputs", exist_ok=True)

def month_agg(m, need_pos):
    """One month partition -> one row per page. Scanned separately and cached, because a
    single 30M-row scan is long enough that a dropped Colab connection kills the whole run."""
    cache = f"work/outputs/_agg_{m}.parquet"
    if os.path.exists(cache):
        d = pd.read_parquet(cache); print(f"  {m}: {len(d)} pages (from cache)"); return d
    extra = (", SUM(gsc_sum_position) AS sumpos,"
             " COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days") if need_pos else ""
    f = f"{BASE}/fact_content_daily_performance/month={m}/data_0.parquet"
    d = con.sql(f"""
        SELECT content_hash_id,
               ANY_VALUE(client_hash_id) AS client_hash_id,
               SUM(gsc_impressions)      AS impr,
               SUM(gsc_clicks)           AS clicks{extra}
        FROM read_parquet('{f}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df()
    d.to_parquet(cache, index=False); print(f"  {m}: {len(d)} pages")
    return d

print("Reading 3 monthly partitions one at a time. Sealed month", SEALED, "is NOT touched.")
jan = month_agg("2026-01", False)
feb = month_agg("2026-02", True)
mar = month_agg("2026-03", False)

jan = jan.rename(columns={"impr": "impr_jan", "clicks": "clicks_jan"}).drop(columns=["client_hash_id"])
mar = mar.rename(columns={"impr": "impr_mar", "clicks": "clicks_mar"}).drop(columns=["client_hash_id"])
feb = feb.rename(columns={"impr": "impr_feb", "clicks": "clicks_feb",
                          "sumpos": "sumpos_feb", "active_days": "active_days_feb"})

p = feb.merge(jan, on="content_hash_id", how="outer").merge(mar, on="content_hash_id", how="outer")
dimc = con.sql(f"""SELECT content_hash_id, content_type, main_intent, word_count
                   FROM read_parquet('{BASE}/dim_content.parquet')""").df()
raw = p.merge(dimc, on="content_hash_id", how="inner")
print("Pages seen in at least one of the three months:", len(raw))


Reading 3 monthly partitions one at a time. Sealed month 2026-06 is NOT touched.
  2026-01: 121544 pages (from cache)
  2026-02: 153559 pages (from cache)
  2026-03: 176738 pages (from cache)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages seen in at least one of the three months: 203074


## 1. Method choice and why

*Which method from the toolkit, and why it fits my lane.*

**The question shape is a ranking, not a yes/no.** An editor has time for a fixed number of pages
this month. The useful output is therefore an ordered queue, and the honest metric is
**precision@K** — of the first K pages I hand over, how many really did under-capture clicks?
The `training-honest-models` skill maps this shape directly: *"which first?" ranking → any
classifier's probability, evaluated at precision@K*. So I train classifiers and rank on their
predicted probability.

**The label (a proxy, and I name it as one).** A page "under-captured clicks in March" if its
March CTR sat in the **bottom quarter of pages in the same February position tier**. Tier-relative,
because comparing a deep page's CTR against a top-3 page's CTR would only rediscover position.
The 25th-percentile cut is computed **inside each training fold** and applied to the held-out fold —
never over the whole dataset. This is a proxy for editorial value, not proof of revenue impact.

**The comparison contract (four parts, from the Week-5 session).** The rule and the model sit the
same exam: **same rows · same folds · same metric + K · same tie policy.** The tie policy matters
more than it sounds for me — in ML-03 I found my rule's top 100 pages all tied at one score, so
ties are a known failure mode of my own baseline, not a footnote. Both are ranked by
`score DESC, then February impressions DESC, then content id ASC` — deterministic, and identical.

**Queue shape: global, not per client.** One editor works one list, so K is counted across all
clients in the fold. A per-client queue would measure a different workload and is not what my
lane describes.

**Models, smallest first.** Logistic Regression (readable coefficients), a depth-3 Decision Tree
(printable), Random Forest, and HistGradientBoosting. HistGB fits this data specifically: it is
built for >10k samples, it handles missing values natively (so I never have to impute, and
`missing -> 0` is exactly the mistake the session called out — *a zero is not "nothing"*), and it
takes categorical features without one-hot encoding.

**Considered and rejected.** Stacking, voting, bagging and AdaBoost — all available, none earn
their interpretability cost here, and the rubric explicitly does not reward complexity alone.
**Monotonic constraints** on position I also rejected, and the reason is my own Week-4 finding:
because the label is defined *within* a position tier, the position-to-CTR relationship I confirmed
in Signal 1 is already largely absorbed by the label definition, so constraining it would not be
justified by the evidence I actually have.

**Permutation importance, not `.feature_importances_`.** scikit-learn warns in two separate places
that impurity-based (MDI) importance is computed on training data and favours high-cardinality
features. I use permutation importance on held-out data instead, with an injected pure-noise
feature as the significance floor.


In [17]:
# ---- Eligibility funnel, then the label definition and its base rate ----
df = raw.copy()

n0 = len(df)
df = df[df["impr_feb"].notna() & df["impr_jan"].notna() & df["impr_mar"].notna()]
n1 = len(df)
df = df[(df["impr_feb"] >= FLOOR_FEB) & (df["impr_mar"] >= FLOOR_MAR)]
n2 = len(df)
df = df[df["sumpos_feb"] > 0]                       # avg_position 0 means "no data", not rank zero
n3 = len(df)
df = df.sort_values("content_hash_id").reset_index(drop=True)   # fixed row order => reproducible noise

print("ELIGIBILITY FUNNEL")
print(f"  pages seen in the window            : {n0}")
print(f"  present in all three months         : {n1}")
print(f"  Feb impressions >= {FLOOR_FEB}, Mar >= {FLOOR_MAR} : {n2}")
print(f"  has real February position data     : {n3}")
print(f"  clients represented                 : {df['client_hash_id'].nunique()}")

# ---- Features: nine numeric fields, all knowable before March, plus content metadata ----
df["ctr_feb"]    = df["clicks_feb"] * 100.0 / df["impr_feb"]     # rate columns are x100 percentages
df["ctr_jan"]    = df["clicks_jan"] * 100.0 / df["impr_jan"].replace(0, np.nan)
df["pos_feb"]    = df["sumpos_feb"] / df["impr_feb"]
df["momentum"]   = df["impr_feb"] / df["impr_jan"].replace(0, np.nan)
df["ctr_mar"]    = df["clicks_mar"] * 100.0 / df["impr_mar"]     # OUTCOME - never a feature
df["noise"]      = rng.normal(size=len(df))                      # significance floor for importance

def tier(p):
    if p <= 3:  return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"
df["tier_feb"] = df["pos_feb"].apply(tier)

df = df.reset_index(drop=True)

print("\nWHAT GOES IN, WHAT STAYS OUT")
print("  IN  (9 numeric, knowable before March): impr_jan, clicks_jan, impr_feb, clicks_feb,")
print("       ctr_jan, ctr_feb, pos_feb, momentum, active_days_feb")
print("  IN  (content metadata)                : word_count, content_type, main_intent")
print("  OUT  March clicks / impressions       : they build the label")
print("  OUT  content id, client id            : grouping keys, not signals")
print("  OUT  sessions, scroll, AI referrals   : only present some of the time")
print("  OUT  trend_direction, trend_pct       : the label trap from the data contract")
print("  NOTE missing values stay NaN          : a zero is not 'nothing'")
print("\nMissing values kept as NaN per column:")
print(df[["ctr_jan","momentum","word_count"]].isna().sum().to_string())

# ---- Base rate: what would a random editor achieve? ----
q25_all = df.groupby("tier_feb")["ctr_mar"].transform(lambda s: s.quantile(0.25))
print(f"\nBASE RATE (whole-panel view, for orientation only): {(df['ctr_mar'] < q25_all).mean():.3f}")
print("Any precision@K at or below this number means the ranking is worthless.")
print("The fold-wise version used for scoring is recomputed inside each training fold.")


ELIGIBILITY FUNNEL
  pages seen in the window            : 203074
  present in all three months         : 97800
  Feb impressions >= 500, Mar >= 100 : 40152
  has real February position data     : 40152
  clients represented                 : 24

WHAT GOES IN, WHAT STAYS OUT
  IN  (9 numeric, knowable before March): impr_jan, clicks_jan, impr_feb, clicks_feb,
       ctr_jan, ctr_feb, pos_feb, momentum, active_days_feb
  IN  (content metadata)                : word_count, content_type, main_intent
  OUT  March clicks / impressions       : they build the label
  OUT  content id, client id            : grouping keys, not signals
  OUT  sessions, scroll, AI referrals   : only present some of the time
  OUT  trend_direction, trend_pct       : the label trap from the data contract
  NOTE missing values stay NaN          : a zero is not 'nothing'

Missing values kept as NaN per column:
ctr_jan           0
momentum          0
word_count    10453

BASE RATE (whole-panel view, for orientation on

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for my question.*

**Two separations, not one.**

**Time.** Every feature comes from January and February; the label comes from March. Nothing that
happened during or after the outcome window can reach the model. The session's exclusion slide is
explicit about why the label-window clicks and impressions are out — *"they build the label."*
Notice what this does **not** ban: February and January clicks are legitimate features, because
they were knowable before the decision. The rule is a temporal boundary, not a blanket ban on
click data.

**Clients.** I use **`GroupKFold` grouped on `client_hash_id`**, so a client's pages are never in
both train and test. Without it, the model can memorise a client's house style and report a score
it will never reproduce on a new client. The session put it plainly: overlapping clients give an
*"overly optimistic evaluation."* `GroupKFold` is deterministic and takes no `random_state`, which
also sidesteps the CV-splitter pitfall in scikit-learn's Common Pitfalls chapter.

**The baseline is refitted inside every fold — this is the subtle one.** My Week-4 rule sets a
page's expected CTR to the median CTR of its position tier. That median is *a statistic learned
from data*. Computing it once over the whole panel would let the rule see the held-out clients'
CTR before being tested on them. Common Pitfalls says exactly this: *"if you have a normalization
step where you divide by the average value, the average should be the average of the train subset,
not the average of all the data."* So the tier medians — and the label's 25th-percentile cuts — are
fitted on the training clients of each fold and only then applied to the held-out clients. The rule
itself is unchanged from Week 4; only *where its numbers come from* is now fold-aware.

**Early stopping is switched off deliberately.** `HistGradientBoosting` enables early stopping by
default above 10,000 samples, and it carves out its internal validation set **at random, with no
knowledge of `client_hash_id`**. That would leak clients inside every fold while I was carefully
holding them out on the outside. I set `early_stopping=False` and choose `max_iter` myself using an
inner grouped split.


In [18]:
# ---- The split, and the fold-aware pieces both competitors share ----
from sklearn.model_selection import GroupKFold

FEATS_NUM  = ["impr_jan","clicks_jan","impr_feb","clicks_feb","ctr_jan","ctr_feb",
              "pos_feb","momentum","active_days_feb","word_count","noise"]
FEATS_CAT  = ["content_type","main_intent"]
FEATS_ALL  = FEATS_NUM + FEATS_CAT
CLICK_FEATS = ["clicks_jan","clicks_feb","ctr_jan","ctr_feb"]   # for the sensitivity run

df[FEATS_NUM] = df[FEATS_NUM].astype("float64")   # pd.NA -> np.nan; keeps missing MISSING
df[FEATS_CAT] = df[FEATS_CAT].astype("object")

groups   = df["client_hash_id"].values
N_SPLITS = 5
gkf      = GroupKFold(n_splits=N_SPLITS)
folds    = list(gkf.split(df, groups=groups))

def fold_label(train_idx, test_idx):
    """25th-percentile CTR cut per February tier, FITTED ON TRAIN ONLY."""
    tr = df.iloc[train_idx]
    cuts = tr.groupby("tier_feb")["ctr_mar"].quantile(0.25)
    glob = tr["ctr_mar"].quantile(0.25)
    y_tr = (tr["ctr_mar"] < tr["tier_feb"].map(cuts).fillna(glob)).astype(int).values
    te = df.iloc[test_idx]
    y_te = (te["ctr_mar"] < te["tier_feb"].map(cuts).fillna(glob)).astype(int).values
    return y_tr, y_te

def fold_rule_scores(train_idx, test_idx):
    """My frozen Week-4 rule, with its tier medians fitted on TRAIN only."""
    tr = df.iloc[train_idx]
    med  = tr.groupby("tier_feb")["ctr_feb"].median()
    glob = tr["ctr_feb"].median()
    te = df.iloc[test_idx]
    expected  = te["tier_feb"].map(med).fillna(glob)
    shortfall = (expected - te["ctr_feb"]).clip(lower=0)
    return (shortfall * te["impr_feb"]).values

print("SPLIT DESIGN")
print(f"  GroupKFold on client_hash_id, n_splits={N_SPLITS}, deterministic (no random_state)")
print(f"  {df['client_hash_id'].nunique()} clients across {len(df)} eligible pages\n")
rows = []
for i, (tr_i, te_i) in enumerate(folds, 1):
    y_tr, y_te = fold_label(tr_i, te_i)
    rows.append({
        "fold": i,
        "train_pages": len(tr_i), "test_pages": len(te_i),
        "train_clients": df.iloc[tr_i]["client_hash_id"].nunique(),
        "test_clients":  df.iloc[te_i]["client_hash_id"].nunique(),
        "overlap_clients": len(set(df.iloc[tr_i]["client_hash_id"]) & set(df.iloc[te_i]["client_hash_id"])),
        "test_base_rate": round(float(y_te.mean()), 3),
    })
fold_tab = pd.DataFrame(rows)
print(fold_tab.to_string(index=False))
assert fold_tab["overlap_clients"].sum() == 0, "client leaked across the split"
print("\nVerified: zero clients appear on both sides of any fold.")
print("\nCAVEAT, stated up front: GroupKFold balances folds by page count, and a few of my 24")
print("clients are large enough to fill a fold alone. Folds 1-3 therefore hold a single test")
print("client each, so their precision@K is measured on one client and is noisy. This is why")
print("I report per-fold numbers and their spread, not just a mean.")


SPLIT DESIGN
  GroupKFold on client_hash_id, n_splits=5, deterministic (no random_state)
  24 clients across 40152 eligible pages

 fold  train_pages  test_pages  train_clients  test_clients  overlap_clients  test_base_rate
    1        29331       10821             23             1                0           0.274
    2        30957        9195             23             1                0           0.302
    3        32606        7546             23             1                0           0.100
    4        33857        6295             15             9                0           0.238
    5        33857        6295             12            12                0           0.166

Verified: zero clients appear on both sides of any fold.

CAVEAT, stated up front: GroupKFold balances folds by page count, and a few of my 24
clients are large enough to fill a fold alone. Folds 1-3 therefore hold a single test
client each, so their precision@K is measured on one client and is noisy. This is

## 3. Train + compare vs my baseline

*Same data, same metric, same split as my Week-4 baseline. Show the table.*

Every model below is a `Pipeline`, so imputation and scaling are fitted on the training fold only
and merely *applied* to the held-out fold — the pipeline is what makes that impossible to forget.
`max_iter` for HistGradientBoosting is chosen by an **inner** grouped split over the training
clients of fold 1, so the outer test folds play no part in the choice.

Then the **sensitivity run**: the whole comparison again with the four click-derived features
removed. The session's slide is the point of it — *"Every method moves. That is how much the
headline leans on click history."* One honest caveat about my own table: unlike the session's
example, **my rule is itself click-derived** (it compares February CTR against a tier median), so
its row is constant because I did not change the rule, not because the rule is click-free.


In [19]:
# ---- Train, rank, compare. One table, both competitors, identical exam. ----
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

KS = [20, 50]

def make_pipe(name, feats, max_iter=200):
    cats = [c for c in feats if c in FEATS_CAT]
    nums = [c for c in feats if c not in FEATS_CAT]
    cat_tf = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="__NA__")),
                       ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    if name == "Logistic":
        num_tf = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                           ("sc", StandardScaler())])
        pre = ColumnTransformer([("c", cat_tf, cats), ("n", num_tf, nums)])
        return Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000))])
    if name == "Tree":
        pre = ColumnTransformer([("c", cat_tf, cats), ("n", SimpleImputer(strategy="median"), nums)])
        return Pipeline([("pre", pre), ("m", DecisionTreeClassifier(max_depth=3, random_state=SEED))])
    if name == "Forest":
        pre = ColumnTransformer([("c", cat_tf, cats), ("n", SimpleImputer(strategy="median"), nums)])
        return Pipeline([("pre", pre), ("m", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED))])
    if name == "HistGB":
        pre = ColumnTransformer([("c", cat_tf, cats), ("n", "passthrough", nums)])
        mask = [True] * len(cats) + [False] * len(nums)
        return Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(
            categorical_features=mask, early_stopping=False, max_iter=max_iter,
            learning_rate=0.06, max_leaf_nodes=31, l2_regularization=1.0, random_state=SEED))])
    raise ValueError(name)

def precision_at_k(test_idx, scores, y_te, ks):
    """ONE tie policy, used by every competitor: score DESC, Feb impressions DESC, id ASC."""
    t = pd.DataFrame({
        "score": scores,
        "impr_feb": df.iloc[test_idx]["impr_feb"].values,
        "cid": df.iloc[test_idx]["content_hash_id"].astype(str).values,
        "y": y_te,
    }).sort_values(["score", "impr_feb", "cid"], ascending=[False, False, True])
    return {k: float(t["y"].head(k).mean()) for k in ks}

# ---- choose max_iter on an INNER grouped split of fold 1's training clients ----
tr_i, _ = folds[0]
inner_df = df.iloc[tr_i]
inner = list(GroupKFold(n_splits=3).split(inner_df, groups=inner_df["client_hash_id"].values))
curve = []
for mi in [50, 100, 200, 400]:
    sc = []
    for a, b in inner:
        ga, gb = tr_i[a], tr_i[b]
        ya, yb = fold_label(ga, gb)
        if ya.sum() == 0 or yb.sum() == 0:
            continue
        p = make_pipe("HistGB", FEATS_ALL, max_iter=mi).fit(df.iloc[ga][FEATS_ALL], ya)
        s = p.predict_proba(df.iloc[gb][FEATS_ALL])[:, 1]
        sc.append(precision_at_k(gb, s, yb, [50])[50])
    curve.append({"max_iter": mi, "inner_precision_at_50": round(float(np.mean(sc)), 3)})
curve = pd.DataFrame(curve)
print("CHOOSING max_iter (inner grouped split, outer test folds never seen):")
print(curve.to_string(index=False))
BEST_ITER = int(curve.loc[curve["inner_precision_at_50"].idxmax(), "max_iter"])
print(f"-> max_iter fixed at {BEST_ITER} for every outer fold.\n")

# ---- the comparison, run twice: with click features and without ----
def run_comparison(feats, tag):
    out = []
    for i, (tr_i, te_i) in enumerate(folds, 1):
        y_tr, y_te = fold_label(tr_i, te_i)
        out.append({"model": "The rule (Week 4)", "fold": i, "base_rate": float(y_te.mean()),
                    **{f"p@{k}": v for k, v in
                       precision_at_k(te_i, fold_rule_scores(tr_i, te_i), y_te, KS).items()}})
        for name in ["Logistic", "Tree", "Forest", "HistGB"]:
            p = make_pipe(name, feats, max_iter=BEST_ITER).fit(df.iloc[tr_i][feats], y_tr)
            s = p.predict_proba(df.iloc[te_i][feats])[:, 1]
            out.append({"model": name, "fold": i, "base_rate": float(y_te.mean()),
                        **{f"p@{k}": v for k, v in precision_at_k(te_i, s, y_te, KS).items()}})
    r = pd.DataFrame(out)
    r["run"] = tag
    return r

full = run_comparison(FEATS_ALL, "with clicks")
noclick_feats = [f for f in FEATS_ALL if f not in CLICK_FEATS]
noclk = run_comparison(noclick_feats, "clicks removed")
allr = pd.concat([full, noclk], ignore_index=True)

order = ["The rule (Week 4)", "Logistic", "Tree", "Forest", "HistGB"]
print("=" * 78)
print("MODEL vs BASELINE - same rows, same folds, same metric, same tie policy")
print("=" * 78)
main = (full.groupby("model")[["base_rate", "p@20", "p@50"]].mean()
            .reindex(order).round(3))
main["p@50 sd across folds"] = full.groupby("model")["p@50"].std().reindex(order).round(3)
print(main.to_string())

print("\nPER-FOLD precision@50 (variability is part of the result):")
print(full.pivot_table(index="model", columns="fold", values="p@50")
          .reindex(order).round(3).to_string())

print("\n" + "=" * 78)
print("SENSITIVITY - the same test with click-derived features removed")
print("=" * 78)
sens = pd.DataFrame({
    "with clicks":    full.groupby("model")["p@50"].mean().reindex(order).round(3),
    "clicks removed": noclk.groupby("model")["p@50"].mean().reindex(order).round(3),
})
sens["move"] = (sens["clicks removed"] - sens["with clicks"]).round(3)
print(sens.to_string())
print("\nThe rule row is constant because I did not change the rule - not because it is click-free.")
print("My rule IS click-derived (Feb CTR vs a tier median), unlike the session's example.")

# ---- the depth-3 tree, printed, because a readable model is worth more than one point ----
p_tree = make_pipe("Tree", FEATS_ALL).fit(df.iloc[folds[0][0]][FEATS_ALL], fold_label(*folds[0])[0])
names = FEATS_CAT + [c for c in FEATS_ALL if c not in FEATS_CAT]
print("\nTHE DEPTH-3 TREE (fold 1), readable end to end:")
print(export_text(p_tree.named_steps["m"], feature_names=names))

metrics = {
    "label": "March CTR in bottom quartile of its February position tier (train-fold cut)",
    "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH, "sealed_month_untouched": SEALED,
    "eligible_pages": int(len(df)), "clients": int(df["client_hash_id"].nunique()),
    "n_splits": N_SPLITS, "max_iter_chosen": BEST_ITER,
    "mean_base_rate": round(float(full["base_rate"].mean()), 3),
    "mean_p_at_50": {m: round(float(v), 3) for m, v in full.groupby("model")["p@50"].mean().items()},
    "mean_p_at_50_clicks_removed": {m: round(float(v), 3) for m, v in noclk.groupby("model")["p@50"].mean().items()},
}
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nReceipts -> work/outputs/w05_model_metrics.json")


CHOOSING max_iter (inner grouped split, outer test folds never seen):
 max_iter  inner_precision_at_50
       50                  0.820
      100                  0.800
      200                  0.833
      400                  0.787
-> max_iter fixed at 200 for every outer fold.

MODEL vs BASELINE - same rows, same folds, same metric, same tie policy
                   base_rate  p@20   p@50  p@50 sd across folds
model                                                          
The rule (Week 4)      0.216  0.56  0.492                 0.107
Logistic               0.216  0.62  0.604                 0.095
Tree                   0.216  0.71  0.692                 0.135
Forest                 0.216  0.74  0.760                 0.089
HistGB                 0.216  0.85  0.784                 0.113

PER-FOLD precision@50 (variability is part of the result):
fold                  1     2     3     4     5
model                                          
The rule (Week 4)  0.50  0.66  0.38  0.50

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Three checks, in order. First **what the model leans on** — permutation importance on held-out
data, with a pure-noise column included as the significance floor: any real feature that cannot
beat random noise is noise. I report `mean ± std` over 20 shuffles and apply the documentation's
own rule, `mean − 2·std > 0`. I check the feature correlations first, because permuting one of two
correlated features leaves the model reading the same information off the other, and both then
look falsely unimportant.

Second, **does the model fix the baseline's known weakness?** In Week 4 my top ten contained seven
effectively zero-click pages — the picks I said the rule was least trustworthy on. If the model is
a real improvement it should put fewer of those at the top of the queue.

Third, **three concrete wrong cases**, printed with their profiles, and an honest sentence about
why they are hard.


In [20]:
# ---- What the model leans on, and where it is wrong ----
from sklearn.inspection import permutation_importance

tr_i, te_i = folds[0]
y_tr, y_te = fold_label(tr_i, te_i)

print("CORRELATION CHECK FIRST (permutation importance is misleading on correlated features)")
corr = df.iloc[te_i][FEATS_NUM].corr().abs()
pairs = [(a, b, round(float(corr.loc[a, b]), 2))
         for i, a in enumerate(FEATS_NUM) for b in FEATS_NUM[i+1:] if corr.loc[a, b] >= 0.7]
print("  pairs with |r| >= 0.70:", pairs if pairs else "none")
if pairs:
    print("  Caveat: importances for these features are depressed for BOTH members of the pair.")

best = make_pipe("HistGB", FEATS_ALL, max_iter=BEST_ITER).fit(df.iloc[tr_i][FEATS_ALL], y_tr)
r = permutation_importance(best, df.iloc[te_i][FEATS_ALL], y_te,
                           n_repeats=20, random_state=SEED, scoring="average_precision", n_jobs=-1)

imp = (pd.DataFrame({"feature": FEATS_ALL, "mean": r.importances_mean, "sd": r.importances_std})
         .sort_values("mean", ascending=False).reset_index(drop=True))
imp["significant (mean-2sd>0)"] = imp["mean"] - 2 * imp["sd"] > 0
noise_row = imp.index[imp["feature"] == "noise"][0]
print("\nPERMUTATION IMPORTANCE (held-out fold 1, average precision, 20 shuffles)")
print(imp.round(4).to_string(index=False))
print(f"\nThe injected pure-noise feature ranks #{noise_row + 1} of {len(imp)}.")
print("Everything below it is indistinguishable from random and should be read as noise.")

# ---- Does the model fix the Week-4 weakness? ----
NEAR_ZERO = 0.02
rule_s  = fold_rule_scores(tr_i, te_i)
model_s = best.predict_proba(df.iloc[te_i][FEATS_ALL])[:, 1]

def top_k_frame(scores, k=50):
    t = df.iloc[te_i].copy()
    t["score"] = scores
    t["y"] = y_te
    return t.sort_values(["score", "impr_feb", "content_hash_id"],
                         ascending=[False, False, True]).head(k)

rule_top, model_top = top_k_frame(rule_s), top_k_frame(model_s)
print("\nDOES THE MODEL FIX THE WEEK-4 WEAKNESS?")
print("  (Week 4: 7 of my top 10 were effectively zero-click pages - my shakiest picks.)")
for nm, t in [("The rule", rule_top), ("HistGB", model_top)]:
    nz = float((t["ctr_feb"] <= NEAR_ZERO).mean())
    nav = float((t["main_intent"].astype(str).str.lower() == "navigational").mean())
    print(f"  {nm:<10} top-50: near-zero-click {nz:.0%} | navigational {nav:.0%} | correct {t['y'].mean():.0%}")

print("\nWHERE THE MODEL IS MOST WRONG (precision@50 picks, by February tier):")
mt = model_top.copy()
print(mt.groupby("tier_feb")["y"].agg(picked="size", correct="sum").assign(
      precision=lambda x: (x["correct"] / x["picked"]).round(2)).to_string())

print("\nTHREE CONCRETE WRONG CASES (ranked high, but did NOT under-capture in March):")
wrong = model_top[model_top["y"] == 0].head(3)
for n, (_, w) in enumerate(wrong.iterrows(), 1):
    print(f"  #{n} tier={w['tier_feb']}, Feb impr={int(w['impr_feb'])}, Feb CTR={w['ctr_feb']:.3f}%, "
          f"Mar CTR={w['ctr_mar']:.3f}%, intent={w['main_intent']}, words={w['word_count']}")
    print(f"      Hard because: February looked weak, but March CTR held up - the page was mid-recovery,")
    print(f"      or February was depressed by a query mix that did not repeat. A one-month feature")
    print(f"      window cannot tell those two apart.")


CORRELATION CHECK FIRST (permutation importance is misleading on correlated features)
  pairs with |r| >= 0.70: [('impr_jan', 'clicks_jan', 0.73), ('impr_jan', 'impr_feb', 0.87), ('impr_jan', 'clicks_feb', 0.7), ('clicks_jan', 'clicks_feb', 0.95)]
  Caveat: importances for these features are depressed for BOTH members of the pair.

PERMUTATION IMPORTANCE (held-out fold 1, average precision, 20 shuffles)
        feature   mean     sd  significant (mean-2sd>0)
        ctr_feb 0.1488 0.0066                      True
        ctr_jan 0.0429 0.0043                      True
     clicks_feb 0.0315 0.0037                      True
       impr_feb 0.0267 0.0047                      True
     word_count 0.0259 0.0034                      True
        pos_feb 0.0232 0.0036                      True
     clicks_jan 0.0078 0.0013                      True
       momentum 0.0048 0.0016                      True
    main_intent 0.0043 0.0012                      True
       impr_jan 0.0040 0.0033    

## What the numbers said

**The model wins, and it wins in every fold.** Against a base rate of 0.216, my frozen Week-4 rule reaches precision@50 of 0.492. HistGradientBoosting reaches 0.784, and beats the rule in all five folds individually — 0.90 / 0.78 / 0.84 / 0.80 / 0.60 against 0.50 / 0.66 / 0.38 / 0.50 / 0.42 — so this is not an average hiding one lucky fold. It wins at the tighter cut too: precision@20 of 0.85 against the rule's 0.56.

**But complexity was not free money.** The depth-3 Decision Tree — a model I can print and read in full — already reaches 0.692, and the Random Forest reaches 0.760 with the lowest fold-to-fold spread of any model (sd 0.089 against HistGB's 0.113). Most of the available lift comes from using the features at all, not from the gradient boosting on top. Boosting buys the last 0.024 at the cost of every scrap of readability, and the inner search chose `max_iter=200` off a curve that is nearly flat (0.820 / 0.800 / 0.833 / 0.787). If someone asked me to defend one model in a room, I would defend the Forest.

**Roughly half the advantage is borrowed from click history.** With the four click-derived features removed, HistGB falls from 0.784 to 0.604 and the Tree collapses from 0.692 to 0.364 — the largest move of any method. Every model drops; the rule's row is flat only because I did not change the rule. The model still clears the rule without clicks (0.604 against 0.492), so the win is real rather than an artefact — but it is a 0.112 margin, not a 0.292 one, and that is the number I would quote.

**A belief of mine from Week 4 was reversed.** I named seven near-zero-click pages in my top ten as the picks I trusted least. The model does the opposite of what I expected: it puts *more* of them at the top of the queue (34% of its top 50, against the rule's 14%) and is right far more often (90% against 50%). Pages that barely converted in February really did under-capture in March. My caution was not supported by this test, and I am recording that rather than quietly dropping it.

**Why the rule loses is partly the metric's fault, and I should say so.** My rule ranks by CTR shortfall *multiplied by impressions* — it was built to put business impact first. precision@K counts hits and is indifferent to whether a hit sat on 500 impressions or 200,000. The comparison honours the four-part contract, but the metric I chose quietly suits the model. A value-weighted metric would be a fairer second exam, and it is the first thing I would add next.

**Two features are dead weight, and one is close.** `content_type` scores exactly 0.0000 — the model never splits on it at all. `impr_jan` fails the significance test (0.0040 ± 0.0033, so mean − 2·sd < 0) and sits barely above the injected noise column. `active_days_feb` passes significance but at 0.0016, which is statistically real and practically nothing. Four feature pairs exceed |r| = 0.70, with `clicks_jan`/`clicks_feb` at 0.95, so importances for those are depressed for both members of each pair and should be read as a floor, not a measurement.

**What the errors look like.** The model makes one recognisable mistake: pages that looked weak in February and recovered in March. All three inspected misses are page-1 pages whose February CTR sat between 0.015% and 0.067% and whose March CTR came back at 0.11–0.16%. A two-month feature window cannot separate "this page genuinely under-converts" from "February was a bad month for this page."

Worth noting where the picks land: **all 50 came from `page_1` (37) or `top_3` (13)** — nothing from striking distance or deeper. Precision is high in both (0.89 and 0.92), so the model is accurate, but it has effectively decided that only already-visible pages are worth an editor's time. That agrees with FlyRank's own published priority action ("improve page-one click capture before rebuilding from scratch"), yet it also means my queue would never surface a striking-distance page, and I did not choose that behaviour — the model did.

**Note on the printed tree:** nearly every leaf shows `class: 0` because the positive class is a minority. Ranking uses `predict_proba`, not the predicted label, so the tree is working as intended and its readable structure is still the point.

**Reproducibility.** Fixing the row order by `content_hash_id` before generating the noise column made consecutive runs identical (same `max_iter`, same importances to four decimals). An earlier version assigned noise by row position, so rebuilding the data pull in a different order silently moved every number — a real defect, found by re-running rather than by reading the code.

**Verdict.** The model beats the frozen baseline on the same rows, folds, metric and tie policy, and it does so in every fold. The honest reading is narrower than the headline: about half the gain rides on click history, the metric I chose flatters the model over the rule, and a readable Random Forest gets within 0.024 of the best score with less variance. I would ship a model over the rule — and I would say all three of those sentences out loud when I did.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymised hash ids
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Baseline and model appear in the **same table**, from the **same notebook run**
- [x] Same rows, same folds, same metric + K, same tie policy — the four-part contract
- [x] Grouped validation on `client_hash_id`; zero client overlap asserted in code
- [x] Time-aware: features from Jan + Feb, label from March, June 2026 sealed and never queried
- [x] Preprocessing, tier medians and label cuts all fitted **inside** the training fold
- [x] `early_stopping=False` so HistGB cannot leak clients through its internal validation split
- [x] Base rate reported next to every precision@K, and per-fold spread shown
- [x] Permutation importance on held-out data with a noise floor — not `.feature_importances_`
- [x] Sensitivity run with click features removed
- [x] Three concrete wrong cases read and explained
- [x] Seed fixed (`SEED = 42`) and stated
- [x] Committed to `work/notebooks/` — then submit the repo URL on the card
